
# Data Extraction (Cleaned, Generalized Picard Pipeline)

This notebook is a streamlined replacement for `Data_Extraction.ipynb`.

What it keeps:
- Parsing Kreuzer-style files in `Example_Files/Y.v06.txt` ... `Y.v27.txt`
- Filtering and processing the smoothable subset (`sing=0` by default)
- CYTools geometry on the small resolution side
- BK data (`dp`, `rk`, `sq`, `Lambda`)
- Wall-data extraction and restriction to Pic(Y)
- Export to JSONL.GZ

What it changes:
- Single generalized rank-`pic` pipeline (no special-case-only `pic=1/2` code path)
- Configurable Picard cutoff (`max_picard`)
- Fewer one-off/debug cells, cleaner output schema


In [31]:

import json
import gzip
import random
import re
import time
from collections import Counter
from functools import reduce
from math import gcd
from pathlib import Path
from itertools import combinations_with_replacement

import numpy as np
import pandas as pd
import sympy as sp
from sympy import Matrix
from sympy.matrices.normalforms import hermite_normal_form
from sympy.polys.domains import GF
from sympy.polys.matrices import DomainMatrix

from cytools import Polytope

CFG = {
    # Input files
    "data_dir": "Example_Files",
    "file_start": 6,
    "file_end": 27,

    # Record filters
    "require_sing0": True,
    "min_picard": 1,
    "max_picard": 2,  # <-- main knob: set highest pic you want to process

    # Optional downsampling for fast testing
    "sample_size": None,
    "random_seed": 0,

    # Progress
    "progress_every": 1000,

    # Kernel saturation (rank-agnostic) tries these primes
    "saturation_primes": [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47],

    # Export
    "export_compressed_jsonl": True,
}

CFG["export_path"] = (
    f"{CFG['data_dir']}/"
    f"wall_data_sing0_pic{CFG['min_picard']}_to_{CFG['max_picard']}_cleaned.jsonl.gz"
)

CFG


{'data_dir': 'Example_Files',
 'file_start': 6,
 'file_end': 27,
 'require_sing0': True,
 'min_picard': 1,
 'max_picard': 2,
 'sample_size': None,
 'random_seed': 0,
 'progress_every': 1000,
 'saturation_primes': [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47],
 'export_compressed_jsonl': True,
 'export_path': 'Example_Files/wall_data_sing0_pic1_to_2_cleaned.jsonl.gz'}

In [32]:

_INT = r"[-+]?\d+"


def _parse_keyvals(header: str):
    out = {}

    m = re.search(rf"\btoric\s*=\s*({_INT})\s*,\s*({_INT})\b", header)
    if m:
        out["toric_h11"] = int(m.group(1))
        out["toric_h21"] = int(m.group(2))

    patterns = {
        "pic": rf"\bpic\s*=\s*({_INT})\b",
        "h12": rf"\bh12\s*=\s*({_INT})\b",
        "E": rf"\bE\s*=\s*({_INT})\b",
        "H3": rf"\bH\^3\s*=\s*({_INT})\b",
        "c2H": rf"\bc2H\s*=\s*({_INT})\b",
        "sing": rf"\bsing\s*=\s*({_INT})\b",
        "rk": rf"\brk\s*=\s*({_INT})\b",
        "sq": rf"(?:^|\s)#sq\s*=\s*({_INT})(?=\s|$)",
        "dp": rf"(?:^|\s)#dp\s*=\s*({_INT})(?=\s|$)",
        "CY_index": rf"(?:^|\s)#CY\s*=\s*({_INT})(?=\s|$)",
    }

    for k, pat in patterns.items():
        mm = re.search(pat, header)
        if mm:
            out[k] = int(mm.group(1))

    return out


def _parse_vertices_block(lines, start_idx):
    if start_idx >= len(lines):
        return None, start_idx

    head = lines[start_idx].strip()
    m = re.match(rf"^\s*({_INT})\s+({_INT})\s+Vertices of P\*\s*\(N-lattice\)", head)
    if not m:
        return None, start_idx

    dim = int(m.group(1))
    nverts = int(m.group(2))

    rows = []
    i = start_idx + 1
    for _ in range(dim):
        if i >= len(lines):
            return None, start_idx
        parts = lines[i].split()
        if len(parts) != nverts:
            return None, start_idx
        rows.append([int(x) for x in parts])
        i += 1

    cols = [[rows[r][j] for r in range(dim)] for j in range(nverts)]

    return {
        "dim": dim,
        "nverts": nverts,
        "vertices_rows": rows,
        "vertices_cols": cols,
        "vertices_header_line": head,
    }, i


def parse_kreuzer_conifold_text(text: str):
    lines = text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    starts = [i for i, ln in enumerate(lines) if ln.strip().startswith("pic=")]
    if not starts:
        return []

    records = []
    for si, start in enumerate(starts):
        end = starts[si + 1] if si + 1 < len(starts) else len(lines)
        block = lines[start:end]

        header = block[0].strip()
        rec = _parse_keyvals(header)
        rec["raw_header"] = header

        vtx = None
        j = 1
        while j < len(block):
            parsed, _ = _parse_vertices_block(block, j)
            if parsed is not None:
                vtx = parsed
                break
            j += 1

        if vtx is not None:
            rec.update(vtx)

        records.append(rec)

    return records


def read_and_parse_kreuzer_file(path: Path):
    txt = path.read_text(encoding="utf-8", errors="ignore")
    return parse_kreuzer_conifold_text(txt)


In [33]:

def iter_input_files(cfg):
    base = Path(cfg["data_dir"])
    for v in range(cfg["file_start"], cfg["file_end"] + 1):
        yield base / f"Y.v{v:02d}.txt"


def load_records(cfg):
    records = []
    missing = []

    for fp in iter_input_files(cfg):
        if not fp.exists():
            missing.append(str(fp))
            continue

        recs = read_and_parse_kreuzer_file(fp)
        for r in recs:
            pic = r.get("pic")
            sing = r.get("sing")

            if pic is None or "vertices_cols" not in r:
                continue
            if cfg["require_sing0"] and sing != 0:
                continue
            if int(pic) < int(cfg["min_picard"]) or int(pic) > int(cfg["max_picard"]):
                continue

            records.append({
                "source_file": str(fp),
                "CY_index": r.get("CY_index"),
                "pic": int(pic),
                "sing": sing,
                "h12": r.get("h12"),
                "E": r.get("E"),
                "H3": r.get("H3"),
                "c2H": r.get("c2H"),
                "toric_h11": r.get("toric_h11"),
                "toric_h21": r.get("toric_h21"),
                "dp": r.get("dp"),
                "rk": r.get("rk"),
                "sq": r.get("sq"),
                "dim": r.get("dim"),
                "nverts": r.get("nverts"),
                "vertices_cols": r["vertices_cols"],
                "raw_header": r.get("raw_header"),
            })

    if cfg["sample_size"] is not None and cfg["sample_size"] < len(records):
        rng = random.Random(cfg["random_seed"])
        records = rng.sample(records, cfg["sample_size"])

    return records, missing


records, missing_files = load_records(CFG)

print("Missing files:", missing_files if missing_files else "None")
print("Loaded records:", len(records))
print("Picard distribution:", dict(sorted(Counter(r["pic"] for r in records).items())))

if records:
    print("Example record keys:", sorted(records[0].keys()))


Missing files: None
Loaded records: 3680
Picard distribution: {1: 210, 2: 3470}
Example record keys: ['CY_index', 'E', 'H3', 'c2H', 'dim', 'dp', 'h12', 'nverts', 'pic', 'raw_header', 'rk', 'sing', 'source_file', 'sq', 'toric_h11', 'toric_h21', 'vertices_cols']


In [34]:

def _maybe_call(x):
    return x() if callable(x) else x


def _as_int_array(x):
    return np.array(x, dtype=int)


def _is_minimal_parallelogram_face(face):
    verts = _as_int_array(_maybe_call(face.vertices))
    if verts.shape[0] != 4:
        return False
    pts = _as_int_array(_maybe_call(face.points))
    return pts.shape[0] == 4


def _parallelogram_pairing(verts4):
    pairings = [(0, 2, 1, 3), (0, 1, 2, 3), (0, 3, 1, 2)]
    for a, c, b, d in pairings:
        if np.all(verts4[a] + verts4[c] == verts4[b] + verts4[d]):
            return a, c, b, d
    return None


def compute_lambda_dp_rk_bk(vertices_cols, verbose=False):
    delta_dual = Polytope(vertices_cols)      # Delta^\circ in N
    delta = delta_dual.dual_polytope()        # Delta in M

    vstar = _as_int_array(_maybe_call(delta_dual.vertices))
    l = vstar.shape[0]
    coord_to_vidx = {tuple(v.tolist()): i for i, v in enumerate(vstar)}

    rows = []
    face_vertex_sets = []

    for face in delta_dual.faces(d=2):
        if not _is_minimal_parallelogram_face(face):
            continue

        verts_face = _as_int_array(_maybe_call(face.vertices))
        pairing = _parallelogram_pairing(verts_face)
        if pairing is None:
            continue

        try:
            idx = [coord_to_vidx[tuple(verts_face[k].tolist())] for k in range(4)]
        except KeyError:
            continue

        a, c, b, d = pairing
        i, j, s, r = idx[a], idx[c], idx[b], idx[d]

        row = np.zeros(l, dtype=int)
        row[i] += 1
        row[j] += 1
        row[s] -= 1
        row[r] -= 1

        nz = np.flatnonzero(row)
        if nz.size > 0 and row[nz[0]] < 0:
            row = -row

        rows.append(row)
        face_vertex_sets.append(tuple(sorted([i, j, s, r])))

    if rows:
        R = np.unique(np.stack(rows, axis=0), axis=0)
    else:
        R = np.zeros((0, l), dtype=int)

    sq = int(R.shape[0])
    rk = int(Matrix(R.tolist()).rank()) if sq > 0 else 0

    def _edge_endpoints(edge):
        ev = _as_int_array(_maybe_call(edge.vertices))
        if ev.shape[0] < 2:
            return None
        return tuple(map(tuple, ev[:2].tolist()))

    def _gcd_list(vals):
        vals = [abs(int(z)) for z in vals]
        return reduce(gcd, vals, 0)

    def _k_theta_from_endpoints(u, v):
        u = np.array(u, dtype=int)
        v = np.array(v, dtype=int)
        return int(_gcd_list((v - u).tolist()))

    edge_map = {}
    for edge in delta.faces(d=1):
        ends = _edge_endpoints(edge)
        if ends is None:
            continue

        u = np.array(ends[0], dtype=int)
        v = np.array(ends[1], dtype=int)

        hits = []
        for w in vstar:
            if int(np.dot(u, w)) == -1 and int(np.dot(v, w)) == -1:
                hits.append(coord_to_vidx[tuple(w.tolist())])

        hits = tuple(sorted(set(hits)))
        if len(hits) == 4:
            edge_map[hits] = (tuple(u.tolist()), tuple(v.tolist()))

    dp = 0
    k_list = []
    missing = 0
    for fvset in face_vertex_sets:
        if fvset not in edge_map:
            missing += 1
            continue
        u, v = edge_map[fvset]
        ktheta = _k_theta_from_endpoints(u, v)
        dp += ktheta
        k_list.append(ktheta)

    if verbose:
        print(f"l={l}, sq={sq}, rk={rk}, dp={dp}, missing_dual_edges={missing}")

    return {
        "Lambda": R,
        "l": l,
        "sq": sq,
        "rk": rk,
        "dp": int(dp),
        "k_list": k_list,
        "missing_dual_edges": int(missing),
    }


In [35]:

def _gcd_int(a, b):
    return int(np.gcd(int(a), int(b)))


def _gcd_vec(v):
    g = 0
    for x in v:
        g = _gcd_int(g, abs(int(x)))
    return g


def _normalize_sign(v):
    v = np.array(v, dtype=int).copy()
    for x in v:
        if x != 0:
            if x < 0:
                v *= -1
            break
    return v


def _primitive(v):
    # Kept as utility for diagnostics; do NOT use it for basis normalization,
    # because dividing by gcd can change the lattice index.
    v = np.array(v, dtype=int).copy()
    g = _gcd_vec(v)
    if g > 1:
        v //= g
    return _normalize_sign(v)


def _clear_denoms_to_int(vec_q):
    denoms = [sp.denom(x) for x in vec_q]
    L = 1
    for d in denoms:
        L = int(sp.ilcm(L, int(d)))
    v = np.array([int(x) for x in list(vec_q * L)], dtype=int)
    return _normalize_sign(v)


def sym_kappa_eval_from_dok(kappa_dok, x, y, z):
    x = np.asarray(x, dtype=int)
    y = np.asarray(y, dtype=int)
    z = np.asarray(z, dtype=int)
    s = 0
    for (i, j, k), val in kappa_dok.items():
        i = int(i)
        j = int(j)
        k = int(k)
        val = int(val)
        if i == j == k:
            s += val * x[i] * y[i] * z[i]
        elif i == j and j < k:
            s += val * (x[i] * y[i] * z[k] + x[i] * y[k] * z[i] + x[k] * y[i] * z[i])
        elif i < j and j == k:
            s += val * (x[i] * y[j] * z[j] + x[j] * y[i] * z[j] + x[j] * y[j] * z[i])
        else:
            s += val * (
                x[i] * y[j] * z[k] + x[i] * y[k] * z[j] +
                x[j] * y[i] * z[k] + x[j] * y[k] * z[i] +
                x[k] * y[i] * z[j] + x[k] * y[j] * z[i]
            )
    return int(s)


def build_Tq_ray_to_pic_overQ(cy):
    h11 = int(cy.h11())
    basis_idx = [int(x) for x in cy.divisor_basis(include_origin=True)]
    if len(basis_idx) != h11:
        raise RuntimeError("divisor_basis length mismatch")

    try:
        Q = cy.glsm_linear_relations(include_origin=True)
    except TypeError:
        Q = cy.glsm_linear_relations()

    Q = np.array(Q, dtype=int)
    Qm = sp.Matrix(Q)

    ns = Qm.nullspace()
    if len(ns) != h11:
        ns2 = sp.Matrix(Q.T).nullspace()
        if len(ns2) != h11:
            raise RuntimeError(f"Expected nullspace dim {h11}, got {len(ns)} (and {len(ns2)} for Q^T)")
        ns = ns2

    Nq = sp.Matrix.hstack(*ns)
    B = Nq.extract(basis_idx, list(range(h11)))
    if B.det() == 0:
        raise RuntimeError("Alignment minor singular")

    Tq = Nq * B.inv()
    if Tq.extract(basis_idx, list(range(h11))) != sp.eye(h11):
        raise RuntimeError("Aligned basis rows are not identity")

    return Tq, basis_idx


def build_S_matrix(cy, triangulation, vertices_cols, Lambda):
    Tq, basis_idx = build_Tq_ray_to_pic_overQ(cy)
    n_rays = int(Tq.rows)

    pts = np.array(triangulation.points(), dtype=int)
    coord_to_ray = {tuple(pts[i].tolist()): i for i in range(pts.shape[0])}

    vstar = np.array(Polytope(vertices_cols).vertices(), dtype=int)
    if Lambda.shape[1] != vstar.shape[0]:
        raise RuntimeError("Lambda column count does not match #vertices of Delta^circ")

    try:
        vert_to_ray = [coord_to_ray[tuple(v.tolist())] for v in vstar]
    except KeyError as e:
        raise RuntimeError(f"Could not map Delta^circ vertex to triangulation ray: {e}")

    rows = []
    for lam in Lambda:
        if not np.any(lam):
            continue

        q = np.zeros(n_rays, dtype=int)
        for j, coeff in enumerate(lam):
            if coeff != 0:
                q[int(vert_to_ray[j])] += int(coeff)

        qv = sp.Matrix([int(x) for x in q]).reshape(n_rays, 1)

        s = qv.extract(basis_idx, [0])
        if Tq * s != qv:
            s = Tq.gauss_jordan_solve(qv)[0]
            if Tq * s != qv:
                raise RuntimeError("Failed to solve Tq*s=q exactly")

        s_int = _clear_denoms_to_int(s)
        if np.any(s_int != 0):
            rows.append(s_int)

    if not rows:
        return np.zeros((0, int(cy.h11())), dtype=int)

    S = np.array(rows, dtype=int)
    S = np.unique(S, axis=0)
    return S



def _reduce_basis_columns(B):
    B = np.array(B, dtype=int)
    if B.ndim == 1:
        B = B.reshape(-1, 1)
    if B.size == 0 or B.shape[1] == 0:
        return np.zeros((B.shape[0], 0), dtype=int)

    # Canonical basis for the COLUMN lattice generated by B.
    # Important: using HNF on B (not B.T) preserves ambient dimension and lets
    # p-saturation replace basis vectors while keeping the same Q-span.
    H = hermite_normal_form(sp.Matrix(B.tolist()))
    H = np.array(H.tolist(), dtype=int)

    if H.ndim == 1:
        H = H.reshape(-1, 1)
    if H.shape[1] == 0:
        return np.zeros((B.shape[0], 0), dtype=int)

    # Primitive/sign-normalize columns for deterministic downstream behavior.
    cols = []
    for j in range(H.shape[1]):
        col = _normalize_sign(H[:, j])
        if np.any(col):
            cols.append(col)

    if not cols:
        return np.zeros((B.shape[0], 0), dtype=int)

    return np.column_stack(cols).astype(int)

def _coeffs_in_basis(B, vec):
    Bm = sp.Matrix(B.tolist())
    vm = sp.Matrix([int(x) for x in vec]).reshape(B.shape[0], 1)
    G = Bm.T * Bm
    rhs = Bm.T * vm
    return G.LUsolve(rhs)


def _in_lattice(B, vec):
    coeff = _coeffs_in_basis(B, vec)
    return all(sp.denom(c) == 1 for c in coeff)


def _rank_mod_p(B, p):
    dm = DomainMatrix.from_Matrix(sp.Matrix(B.tolist())).convert_to(GF(p))
    return int(dm.rank())


def _nullspace_mod_p(B, p):
    dm = DomainMatrix.from_Matrix(sp.Matrix(B.tolist())).convert_to(GF(p))
    ns = dm.nullspace().to_Matrix()  # shape: (nullity, ncols)
    vecs = []
    for i in range(ns.rows):
        v = np.array([int(ns[i, j]) % p for j in range(ns.cols)], dtype=int)
        if np.any(v % p):
            vecs.append(v)
    return vecs


def _p_saturate(B, p, max_iter=10):
    B = np.array(B, dtype=int)
    changed = False

    for _ in range(max_iter):
        r = B.shape[1]
        if _rank_mod_p(B, p) == r:
            break

        additions = []
        for c in _nullspace_mod_p(B, p):
            prod = B @ c
            if np.any(prod % p):
                continue
            cand = (prod // p).astype(int)
            if not _in_lattice(B, cand):
                additions.append(_normalize_sign(cand))

        if not additions:
            break

        B = np.column_stack([B] + [a.reshape(-1, 1) for a in additions])
        B = _reduce_basis_columns(B)
        changed = True

    return B, changed



def saturate_basis_columns(B, primes, max_rounds=4):
    B = _reduce_basis_columns(np.array(B, dtype=int))
    sat_steps = []

    for round_i in range(max_rounds):
        if B.shape[1] == 0:
            break

        any_change = False
        for p in primes:
            before_rank = _rank_mod_p(B, p)
            if before_rank == B.shape[1]:
                continue

            B2, changed = _p_saturate(B, p)
            after_rank = _rank_mod_p(B2, p)

            sat_steps.append({
                "round": int(round_i + 1),
                "p": int(p),
                "rank_mod_p_before": int(before_rank),
                "rank_mod_p_after": int(after_rank),
                "changed": bool(changed),
            })

            if changed:
                any_change = True
                B = _reduce_basis_columns(B2)

        if not any_change:
            break

    return B, sat_steps

def kernel_basis_from_S(S, expected_dim, saturation_primes):
    h11 = int(S.shape[1]) if S.size else int(expected_dim)

    if S.shape[0] == 0:
        B = np.eye(h11, dtype=int)
    else:
        Sm = sp.Matrix(S.tolist())
        ns = Sm.nullspace()
        if len(ns) != expected_dim:
            raise RuntimeError(f"Expected ker(S) dim {expected_dim}, got {len(ns)}")

        cols = [_clear_denoms_to_int(v) for v in ns]
        B = np.column_stack(cols)

    B = _reduce_basis_columns(B)
    if B.shape[1] != expected_dim:
        raise RuntimeError(f"Kernel basis rank mismatch before saturation: {B.shape[1]} vs expected {expected_dim}")

    B_sat, sat_steps = saturate_basis_columns(B, saturation_primes)
    B_sat = _reduce_basis_columns(B_sat)

    if B_sat.shape[1] != expected_dim:
        raise RuntimeError(f"Kernel basis rank mismatch after saturation: {B_sat.shape[1]} vs expected {expected_dim}")

    return B_sat, sat_steps


def compute_small_resolution_data(cy, toric_variety):
    h11 = int(cy.h11())
    h21 = int(cy.h21())

    div_basis_idx = [int(x) for x in cy.divisor_basis(include_origin=True)]
    kappa_dok = cy.intersection_numbers(in_basis=True, format="dok")
    kappa_items = []
    for (a, b, c), val in kappa_dok.items():
        kappa_items.append([int(a), int(b), int(c), int(val)])
    kappa_items.sort()

    c2_vec = [int(x) for x in cy.second_chern_class(in_basis=True, include_origin=False)]

    return {
        "ambient_smooth": bool(toric_variety.is_smooth()),
        "ambient_K_smooth": bool(toric_variety.canonical_divisor_is_smooth()),
        "cy_smooth": bool(cy.is_smooth()),
        "h11": h11,
        "h21": h21,
        "chi": int(cy.chi()),
        "b2": h11,
        "b3": int(2 * h21 + 2),
        "divisor_basis_indices_include_origin": div_basis_idx,
        "kappa_items_in_basis": kappa_items,
        "c2_dot_D_in_basis": c2_vec,
    }


def restrict_to_pic_basis(cy, basis_cols):
    r = basis_cols.shape[1]
    kappa_dok = cy.intersection_numbers(in_basis=True, format="dok")
    c2_vec = np.array([int(x) for x in cy.second_chern_class(in_basis=True, include_origin=False)], dtype=int)

    kappa_pic = []
    for a, b, c in combinations_with_replacement(range(r), 3):
        val = sym_kappa_eval_from_dok(kappa_dok, basis_cols[:, a], basis_cols[:, b], basis_cols[:, c])
        if val != 0:
            kappa_pic.append([int(a), int(b), int(c), int(val)])

    c2_pic = [int(np.dot(c2_vec, basis_cols[:, a])) for a in range(r)]

    return kappa_pic, c2_pic



def infer_pic1_scale_and_sign(H3_hdr, c2H_hdr, H3_base, c2_base, max_k=64):
    if H3_hdr is None or c2H_hdr is None:
        return None, None

    H3_hdr = int(H3_hdr)
    c2H_hdr = int(c2H_hdr)
    H3_base = int(H3_base)
    c2_base = int(c2_base)

    # sign=+1 means keep basis, sign=-1 means use H -> -H
    for sign in (1, -1):
        Hb = sign * H3_base
        cb = sign * c2_base
        for k in range(1, max_k + 1):
            if (k ** 3) * Hb == H3_hdr and k * cb == c2H_hdr:
                return int(k), int(sign)

    return None, None

def process_record(record, cfg):
    V = record["vertices_cols"]

    poly = Polytope(V)
    triang = poly.triangulate()
    cy = triang.get_cy()
    toric_v = triang.get_toric_variety()

    small = compute_small_resolution_data(cy, toric_v)
    bk = compute_lambda_dp_rk_bk(V, verbose=False)

    S = build_S_matrix(cy, triang, V, np.array(bk["Lambda"], dtype=int))

    expected_pic = int(record["pic"])
    basis_cols, sat_steps = kernel_basis_from_S(
        S=S,
        expected_dim=expected_pic,
        saturation_primes=cfg["saturation_primes"],
    )

    rankS = int(Matrix(S.tolist()).rank()) if S.size else 0
    kappa_pic, c2_pic = restrict_to_pic_basis(cy, basis_cols)



    pic1_scale = None
    pic1_basis_sign = None
    if expected_pic == 1 and len(c2_pic) == 1:
        H3_base = int(kappa_pic[0][3]) if kappa_pic else 0
        k_try, sign_try = infer_pic1_scale_and_sign(
            H3_hdr=record.get("H3"),
            c2H_hdr=record.get("c2H"),
            H3_base=H3_base,
            c2_base=c2_pic[0],
        )

        if sign_try == -1:
            basis_cols[:, 0] *= -1
            kappa_pic, c2_pic = restrict_to_pic_basis(cy, basis_cols)
            H3_base = int(kappa_pic[0][3]) if kappa_pic else 0

        if sign_try is None:
            # fallback deterministic orientation when header matching is unavailable
            # or inconclusive: prefer c2 >= 0.
            if c2_pic[0] < 0:
                basis_cols[:, 0] *= -1
                kappa_pic, c2_pic = restrict_to_pic_basis(cy, basis_cols)
                H3_base = int(kappa_pic[0][3]) if kappa_pic else 0

            k_try, sign_try = infer_pic1_scale_and_sign(
                H3_hdr=record.get("H3"),
                c2H_hdr=record.get("c2H"),
                H3_base=H3_base,
                c2_base=c2_pic[0],
            )

        pic1_scale = k_try
        pic1_basis_sign = sign_try if sign_try is not None else 1

    return {

        "meta": {
            "source_file": record.get("source_file"),
            "CY_index": record.get("CY_index"),
            "pic": expected_pic,
            "sing": record.get("sing"),
            "h12": record.get("h12"),
            "E": record.get("E"),
            "H3_header": record.get("H3"),
            "c2H_header": record.get("c2H"),
            "raw_header": record.get("raw_header"),
        },
        "xhat": {
            "vertices_cols": V,
            "nverts": record.get("nverts"),
            "dp_header": record.get("dp"),
            "rk_header": record.get("rk"),
            "sq_header": record.get("sq"),
            "dp_bk": int(bk["dp"]),
            "rk_bk": int(bk["rk"]),
            "sq_bk": int(bk["sq"]),
            "missing_dual_edges_bk": int(bk["missing_dual_edges"]),
            **small,
        },
        "y": {
            "pic": expected_pic,
            # columns are Pic(Y) basis vectors written in Pic(Xhat) divisor basis
            "basis_cols_in_xhat_basis": basis_cols.tolist(),
            "kappa_items_in_pic_basis": kappa_pic,
            "c2_in_pic_basis": c2_pic,
            "pic1_scale_k_against_header": pic1_scale,
            "pic1_basis_sign": pic1_basis_sign,
        },
        "checks": {
            "hodge_header_vs_cytools_match": (
                None
                if record.get("toric_h11") is None or record.get("toric_h21") is None
                else [int(record.get("toric_h11")), int(record.get("toric_h21"))] == [int(small["h11"]), int(small["h21"])]
            ),
            "bk_header_match": (
                None
                if record.get("dp") is None or record.get("rk") is None or record.get("sq") is None
                else [int(record.get("dp")), int(record.get("rk")), int(record.get("sq"))] == [int(bk["dp"]), int(bk["rk"]), int(bk["sq"])]
            ),
            "rankS": int(rankS),
            "h11_minus_rankS": int(small["h11"] - rankS),
            "pic_expected": int(expected_pic),
            "kernel_saturation_steps": sat_steps,
        },
    }


In [36]:

results = []
errors = []

start = time.time()
N = len(records)

for i, rec in enumerate(records, start=1):
    tag = f"{rec.get('source_file')}::CY={rec.get('CY_index')}"
    try:
        out = process_record(rec, CFG)
        results.append(out)
    except Exception as e:
        errors.append({
            "source_file": rec.get("source_file"),
            "CY_index": rec.get("CY_index"),
            "pic": rec.get("pic"),
            "error": repr(e),
            "raw_header": rec.get("raw_header"),
        })

    if (i % CFG["progress_every"] == 0) or (i == N):
        elapsed = time.time() - start
        rate = i / elapsed if elapsed > 0 else 0.0
        print(f"[{i}/{N}] ok={len(results)} err={len(errors)} rate={rate:.2f} rec/s")

elapsed = time.time() - start
print("DONE")
print("records processed:", N)
print("ok:", len(results))
print("errors:", len(errors))
print(f"elapsed: {elapsed:.1f}s")


[1000/3680] ok=1000 err=0 rate=37.99 rec/s
[2000/3680] ok=2000 err=0 rate=28.42 rec/s
[3000/3680] ok=3000 err=0 rate=22.87 rec/s
[3680/3680] ok=3680 err=0 rate=19.99 rec/s
DONE
records processed: 3680
ok: 3680
errors: 0
elapsed: 184.1s


In [37]:

if results:
    rows = []
    for r in results:
        rows.append({
            "pic": r["meta"]["pic"],
            "source_file": r["meta"]["source_file"],
            "CY_index": r["meta"]["CY_index"],
            "h11": r["xhat"]["h11"],
            "h21": r["xhat"]["h21"],
            "rk_bk": r["xhat"]["rk_bk"],
            "sq_bk": r["xhat"]["sq_bk"],
            "dp_bk": r["xhat"]["dp_bk"],
            "bk_match": r["checks"]["bk_header_match"],
            "hodge_match": r["checks"]["hodge_header_vs_cytools_match"],
            "rankS": r["checks"]["rankS"],
            "h11_minus_rankS": r["checks"]["h11_minus_rankS"],
            "pic_expected": r["checks"]["pic_expected"],
            "kernel_dim_match": r["checks"]["h11_minus_rankS"] == r["checks"]["pic_expected"],
            "pic1_scale_k": r["y"].get("pic1_scale_k_against_header"),
        })

    summary_df = pd.DataFrame(rows)

    print("Picard counts:")
    print(summary_df["pic"].value_counts().sort_index())

    print("\nKernel dimension mismatches:", int((~summary_df["kernel_dim_match"]).sum()))

    if summary_df["bk_match"].notna().any():
        bad_bk = int((summary_df["bk_match"] == False).sum())
        print("BK mismatches where header present:", bad_bk)

    if summary_df["hodge_match"].notna().any():
        bad_hodge = int((summary_df["hodge_match"] == False).sum())
        print("Hodge mismatches where header present:", bad_hodge)

    pic1_df = summary_df[summary_df["pic"] == 1]
    if len(pic1_df) > 0:
        print("\nPic=1 scale-k distribution (header-vs-base):")
        print(pic1_df["pic1_scale_k"].value_counts(dropna=False).sort_index())


Picard counts:
pic
1     210
2    3470
Name: count, dtype: int64

Kernel dimension mismatches: 0
BK mismatches where header present: 0
Hodge mismatches where header present: 0

Pic=1 scale-k distribution (header-vs-base):
pic1_scale_k
1.0    184
2.0     23
3.0      1
6.0      2
Name: count, dtype: int64


In [38]:

out_path = Path(CFG["export_path"])
out_path.parent.mkdir(parents=True, exist_ok=True)

if CFG["export_compressed_jsonl"]:
    with gzip.open(out_path, "wt", encoding="utf-8") as f:
        for rec in results:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
else:
    plain_path = Path(str(out_path).replace(".gz", ""))
    with plain_path.open("w", encoding="utf-8") as f:
        for rec in results:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    out_path = plain_path

err_path = Path(str(out_path).replace(".jsonl.gz", "_errors.jsonl").replace(".jsonl", "_errors.jsonl"))
with err_path.open("w", encoding="utf-8") as f:
    for rec in errors:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

summary_path = Path(str(out_path).replace(".jsonl.gz", "_summary.csv").replace(".jsonl", "_summary.csv"))
if "summary_df" in globals():
    summary_df.to_csv(summary_path, index=False)

print("Wrote:")
print(" -", out_path)
print(" -", err_path)
if "summary_df" in globals():
    print(" -", summary_path)


Wrote:
 - Example_Files/wall_data_sing0_pic1_to_2_cleaned.jsonl.gz
 - Example_Files/wall_data_sing0_pic1_to_2_cleaned_errors_errors.jsonl
 - Example_Files/wall_data_sing0_pic1_to_2_cleaned_summary.csv
